# 扩散模型

```python
self.register_buffer(
    "alpha_bars",
    torch.tensor([torch.prod(self.alphas[: i + 1]) for i in range(len(self.alphas))]),  # 累乘
)
```
- `register_buffer` 用于在 PyTorch 的 `nn.Module` 中注册一个不会被优化器更新的持久变量，常用于保存模型推理/训练过程中需要的中间量。
- `alpha_bars` 是扩散模型中的累计噪声衰减因子，公式为 $\bar{\alpha}_t = \prod_{i=1}^t \alpha_i$，表示从第 1 步到第 t 步所有 $\alpha$ 的连乘积。
- `torch.prod(self.alphas[: i + 1])` 计算从第 0 步到第 i 步的 $\alpha$ 连乘，结果组成一个长度为 T 的张量，便于后续采样和反向过程调用。
- 这种写法将公式 $\bar{\alpha}_t$ 直接转化为 Python 列表推导式，体现了公式与代码的紧密对应关系。

```python
if self.scheduling == "linear":
    # 线性调度: β_t 从 beta_min 线性增长到 beta_max
    # beta_min: 初始噪声水平 (通常 ~1e-4)
    # beta_max: 最终噪声水平 (通常 ~0.02)
    self.beta_min, self.beta_max = cfg["beta_min"], cfg["beta_max"]
    
    # 生成 M 个均匀分布的 β 值
    # betas shape: [M]
    # 例如: M=1000 时, betas[0]=0.0001, betas[999]=0.02
    betas = torch.linspace(self.beta_min, self.beta_max, self.M)
else:
# ------------------------- #

```
t 是查表索引，$\alpha$ 是实际控制噪声量的数值。两者的关系链：

$$
t \xrightarrow{\text{索引}} \beta_t \xrightarrow{\alpha_t = 1 - \beta_t} \alpha_t \xrightarrow{\text{累计连乘}} \bar{\alpha}_t = \prod_{i=0}^{t} \alpha_i
$$

- $t$：索引  
- $\beta_t$：噪声参数  
- $\alpha_t = 1 - \beta_t$：每步噪声保留因子  
- $\bar{\alpha}_t = \prod_{i=0}^{t} \alpha_i$：累计噪声保留因子
